# 00 — Earth Engine re-extraction (v2 inputs)

Replaces the lost extraction script behind every v1 CSV. It builds the grid
locally from the district boundaries, asks Earth Engine to reduce each layer
over the cells in resumable chunks, writes the results straight into
`data/v2/`, and refuses the output unless it passes physical
checks. `02_carbon_pipeline.ipynb` then runs in v2 (census) mode automatically.

| v1 defect (AUDIT.md) | Fix here |
|---|---|
| F1 masked SoilGrids pixels averaged as zeros | mask before reducing; valid fraction exported per cell |
| F2 `Grid_ID` random | `cell_id = r{row}_c{col}` from the lattice, identical to `config.cell_key` |
| F3 MOD17 scale factor skipped | DN masked to its valid range and × 0.1 → gC/m²/yr in Earth Engine |
| F5 SOC only for a sample | SOC for every cell |
| F7 depth interval unrecorded | thickness-weighted 0–30 cm; `cfvo`; SoilGrids `ocs` cross-check |
| F9 grid not clipped | grid generated from the boundaries; per-cell fraction inside each |
| NDVI undocumented, median-filled | Sentinel-2 monthly medians with clear-observation counts |

**Boundaries** (`data/boundaries/`): geoBoundaries gbOpen ADM2
(default, 369,961 ha, CC BY 4.0) and Datameet Census 2011 (358,482 ha, ODbL,
sensitivity). Census 2011 reports 376,700 ha.

**Run:** set `EE_PROJECT`, then run all. First run asks you to authenticate in
the browser. Interrupted runs resume from the last finished chunk.

In [1]:
import json
import sys
import time
from datetime import date
from pathlib import Path

import ee
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config as C

EE_PROJECT  = "my-projects-510917"
NPP_YEAR    = 2024              # MOD17A3HGF annual composite
NDVI_START  = "2024-06-01"      # 12 months, Jun 2024 - May 2025
NDVI_MONTHS = 12
CHUNK       = 1500              # cells per Earth Engine request
SCALE = {"soil": 50, "npp": 50, "landcover": 10, "terrain": 30, "ndvi": 20}

V2 = C.DATA / "v2"
PARTS = V2 / "chunks"
PARTS.mkdir(parents=True, exist_ok=True)

In [2]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine", ee.__version__, "| project", EE_PROJECT, "| test:", ee.Number(1).add(1).getInfo())

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine 1.7.47 | project my-projects-510917 | test: 2


## 1. Grid (local, no Earth Engine)

In [3]:
grid = C.build_grid_v2()
grid.to_csv(V2 / "v2_grid.csv", index=False)
for b in [C.BOUNDARY_DEFAULT, C.BOUNDARY_CHECK]:
    print(f"{b:34s} area inside {(grid.cell_area_ha * grid[f'in_{b}']).sum():>10,.0f} ha")
print(f"{len(grid):,} cells touch either boundary")

ludhiana_geoboundaries_adm2        area inside    369,961 ha
ludhiana_census2011_datameet       area inside    358,482 ha
71,197 cells touch either boundary


## 2. Images

Every band is masked where its source has no valid data, so a cell mean is
taken over valid pixels only; `*_valid_frac` records how much of the cell was
valid.

In [4]:
bnd = C.load_boundary(C.BOUNDARY_DEFAULT).union(C.load_boundary(C.BOUNDARY_CHECK))
region = ee.Geometry.Rectangle(list(bnd.bounds))

# SoilGrids 2.0, 0-30 cm thickness-weighted
DEPTHS, WEIGHTS = ["0-5cm", "5-15cm", "15-30cm"], [5, 10, 15]
FACTORS = {"soc": 10, "bdod": 100, "sand": 10, "clay": 10, "cfvo": 10}
bd_top = ee.Image("projects/soilgrids-isric/bdod_mean").select("bdod_0-5cm_mean")
soil_valid = bd_top.mask().And(bd_top.gt(0)).unmask(0)

def soilgrids_030(prop):
    img = ee.Image(f"projects/soilgrids-isric/{prop}_mean")
    acc = None
    for d, w in zip(DEPTHS, WEIGHTS):
        b = img.select(f"{prop}_{d}_mean").multiply(w)
        acc = b if acc is None else acc.add(b)
    return acc.divide(sum(WEIGHTS)).divide(FACTORS[prop]).rename(f"{prop}_030")

ocs = ee.Image("projects/soilgrids-isric/ocs_mean")
print("ocs bands:", ocs.bandNames().getInfo())
soil = (ee.Image.cat([soilgrids_030(p) for p in FACTORS] + [ocs.select(0).rename("ocs_030_t_ha")])
        .updateMask(soil_valid).addBands(soil_valid.rename("soil_valid_frac")))

# MODIS MOD17A3HGF v6.1
mod17 = ee.ImageCollection("MODIS/061/MOD17A3HGF").filterDate(f"{NPP_YEAR}-01-01", f"{NPP_YEAR + 1}-01-01")
if mod17.size().getInfo() == 0:
    raise ValueError(f"No MOD17A3HGF composite for {NPP_YEAR}")
mod17 = mod17.first()
dn = mod17.select("Npp")
npp_ok = dn.gte(-30000).And(dn.lte(32700))
npp = (dn.updateMask(npp_ok).multiply(0.1).rename("npp_gc_m2_yr")
       .addBands(mod17.select("Npp_QC").updateMask(npp_ok).rename("npp_qc"))
       .addBands(npp_ok.unmask(0).rename("npp_valid_frac")))

# ESA WorldCover 2021 and SRTM
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
landcover = ee.Image.cat([wc.eq(40).rename("cropland_frac"), wc.eq(50).rename("builtup_frac"),
                         wc.eq(80).rename("water_frac"), wc.eq(10).rename("tree_frac")])
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
terrain = ee.Image.cat([dem.rename("dem_m"), ee.Terrain.slope(dem).rename("slope_deg")])

# Sentinel-2 monthly NDVI medians with clear-observation counts
CLEAR_SCL = [4, 5, 6, 7]
def month_ndvi(i):
    start = ee.Date(NDVI_START).advance(i, "month")
    tag = start.format("MMMYY").getInfo()
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(region)
           .filterDate(start, start.advance(1, "month"))
           .map(lambda im: im.updateMask(im.select("SCL").remap(CLEAR_SCL, [1] * 4, 0))
                             .normalizedDifference(["B8", "B4"])))
    return col.median().rename(f"NDVI_{tag}"), col.count().rename(f"nobs_{tag}")
pairs = [month_ndvi(i) for i in range(NDVI_MONTHS)]
ndvi = ee.Image.cat([p[0] for p in pairs] + [p[1] for p in pairs])

IMAGES = {"soil": soil, "npp": npp, "landcover": landcover, "terrain": terrain, "ndvi": ndvi}
for k, im in IMAGES.items():
    print(f"{k:10s} {im.bandNames().getInfo()}")

ocs bands: ['ocs_0-30cm_mean']


soil       ['soc_030', 'bdod_030', 'sand_030', 'clay_030', 'cfvo_030', 'ocs_030_t_ha', 'soil_valid_frac']


npp        ['npp_gc_m2_yr', 'npp_qc', 'npp_valid_frac']


landcover  ['cropland_frac', 'builtup_frac', 'water_frac', 'tree_frac']


terrain    ['dem_m', 'slope_deg']


ndvi       ['NDVI_Jun24', 'NDVI_Jul24', 'NDVI_Aug24', 'NDVI_Sep24', 'NDVI_Oct24', 'NDVI_Nov24', 'NDVI_Dec24', 'NDVI_Jan25', 'NDVI_Feb25', 'NDVI_Mar25', 'NDVI_Apr25', 'NDVI_May25', 'nobs_Jun24', 'nobs_Jul24', 'nobs_Aug24', 'nobs_Sep24', 'nobs_Oct24', 'nobs_Nov24', 'nobs_Dec24', 'nobs_Jan25', 'nobs_Feb25', 'nobs_Mar25', 'nobs_Apr25', 'nobs_May25']


## 3. Reduce over the cells (resumable)

Each request sends 1,500 cell rectangles and gets back their means as a
DataFrame. Finished chunks are saved under `v2/chunks/`, so re-running this
cell continues where it stopped. Expect roughly 30–90 minutes in total,
mostly for NDVI.

In [5]:
def cell_fc(df):
    s = C.GRID_STEP_DEG
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Rectangle([c * s, r * s, (c + 1) * s, (r + 1) * s], None, False), {"cell_id": k})
        for k, r, c in zip(df.cell_id, df.grid_row, df.grid_col)])

def reduce_chunk(name, df, tries=4):
    img = IMAGES[name]
    fc = img.reduceRegions(collection=cell_fc(df), reducer=ee.Reducer.mean(), scale=SCALE[name], tileScale=4)
    for attempt in range(tries):
        try:
            out = ee.data.computeFeatures({"expression": fc, "fileFormat": "PANDAS_DATAFRAME"})
            cols = ["cell_id"] + img.bandNames().getInfo()
            return out.reindex(columns=cols)
        except Exception as exc:
            wait = 15 * 2 ** attempt
            print(f"   {name}: {type(exc).__name__}: {str(exc)[:120]} — retry in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"{name} chunk failed {tries} times")

chunks = [grid.iloc[i:i + CHUNK] for i in range(0, len(grid), CHUNK)]
t0 = time.time()
for name in IMAGES:
    for j, df in enumerate(chunks):
        part = PARTS / f"{name}_{j:03d}.csv"
        if part.exists():
            continue
        t = time.time()
        reduce_chunk(name, df).to_csv(part, index=False)
        print(f"{name:10s} chunk {j + 1:2d}/{len(chunks)}  {time.time() - t:5.1f}s  (elapsed {(time.time() - t0) / 60:.1f} min)")
    tab = pd.concat([pd.read_csv(p) for p in sorted(PARTS.glob(f"{name}_*.csv"))], ignore_index=True)
    assert tab.cell_id.is_unique and len(tab) == len(grid), f"{name}: {len(tab)} rows for {len(grid)} cells"
    tab.to_csv(V2 / f"v2_{name}.csv", index=False)
    print(f"== v2_{name}.csv  {len(tab):,} rows")

== v2_soil.csv  71,197 rows


== v2_npp.csv  71,197 rows


== v2_landcover.csv  71,197 rows


== v2_terrain.csv  71,197 rows


== v2_ndvi.csv  71,197 rows


In [6]:
manifest = {
    "run_date": str(date.today()), "ee_project": EE_PROJECT, "ee_version": ee.__version__,
    "boundaries": {"default": C.BOUNDARY_DEFAULT, "check": C.BOUNDARY_CHECK},
    "n_cells": len(grid), "grid_step_deg": C.GRID_STEP_DEG, "chunk": CHUNK, "scales_m": SCALE,
    "soilgrids": {"assets": "projects/soilgrids-isric/{prop}_mean", "depths": DEPTHS, "weights_cm": WEIGHTS,
                  "factors": FACTORS, "ocs": "projects/soilgrids-isric/ocs_mean (band 0)"},
    "npp": {"asset": "MODIS/061/MOD17A3HGF", "year": NPP_YEAR, "band": "Npp", "valid_range": [-30000, 32700], "to_gc_m2_yr": 0.1},
    "landcover": "ESA/WorldCover/v200 (2021): 10 tree, 40 cropland, 50 built-up, 80 water",
    "terrain": "USGS/SRTMGL1_003 elevation; ee.Terrain.slope",
    "ndvi": {"collection": "COPERNICUS/S2_SR_HARMONIZED", "start": NDVI_START, "months": NDVI_MONTHS,
             "clear_scl": CLEAR_SCL, "composite": "median"},
    "reducer": "mean over unmasked pixels; *_valid_frac = mean of the validity mask",
}
(V2 / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("manifest written")

manifest written


## 4. Validate and assemble `ludhiana_cells_v2.csv.gz`

Every check must pass; a failure means the extraction needs attention.

In [7]:
tables = {n: pd.read_csv(V2 / f"v2_{n}.csv") for n in ["grid"] + list(IMAGES)}
cells = tables["grid"]
for n in IMAGES:
    cells = cells.merge(tables[n], on="cell_id", how="left", validate="one_to_one")

checks = {}
full = cells[cells["soil_valid_frac"] >= 0.999]
checks["bulk density 0.9-1.9 g/cm3 where valid"] = full["bdod_030"].between(0.9, 1.9).all()
checks["no zero soil values where valid"] = (full[["soc_030", "bdod_030", "sand_030", "clay_030"]] > 0).all().all()
checks["sand + clay <= 100 %"] = (full["sand_030"] + full["clay_030"] <= 100).all()
checks["sand x clay correlation negative"] = full["sand_030"].corr(full["clay_030"]) < 0
checks["masked cells are NaN, not zero"] = cells.loc[cells["soil_valid_frac"] == 0, "soc_030"].isna().all()
checks["NPP inside MOD17 scaled range"] = cells["npp_gc_m2_yr"].dropna().between(-3000, 3270).all()
checks["in_district_frac within [0, 1]"] = cells["in_district_frac"].between(0, 1).all()
checks["SoilGrids ocs 5-150 t/ha where valid"] = full["ocs_030_t_ha"].between(5, 150).all()
r_ocs = full["ocs_030_t_ha"].corr(full["soc_030"] * full["bdod_030"])
checks["ocs and SOC x BD positively correlated"] = r_ocs > 0
# Diagnostic, not a pass/fail check: SoilGrids models ocs directly, so it need not equal SOC x BD x 30.
ratio = (C.soc_stock_tc_ha(full["soc_030"], full["bdod_030"], 30, full["cfvo_030"] / 100) / full["ocs_030_t_ha"]).median()

for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
print(f"\nmedian computed stock (SOC x BD x 30 x (1 - cfvo)) / SoilGrids ocs = {ratio:.3f}  (diagnostic); r = {r_ocs:.2f}")
print(f"cells with valid soil: {(cells.soil_valid_frac > 0).mean():.1%} | with MOD17: {(cells.npp_valid_frac > 0).mean():.1%}")
assert all(checks.values()), "v2 inputs failed validation; do not use them"
cells.to_csv(V2 / "ludhiana_cells_v2.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})  # compressed so it can be committed
print("Wrote data/v2/ludhiana_cells_v2.csv.gz", f"({len(cells):,} cells). 02_carbon_pipeline.ipynb now runs in v2 mode.")

PASS bulk density 0.9-1.9 g/cm3 where valid
PASS no zero soil values where valid
PASS sand + clay <= 100 %
PASS sand x clay correlation negative
PASS masked cells are NaN, not zero
PASS NPP inside MOD17 scaled range
PASS in_district_frac within [0, 1]
PASS SoilGrids ocs 5-150 t/ha where valid
PASS ocs and SOC x BD positively correlated

median computed stock (SOC x BD x 30 x (1 - cfvo)) / SoilGrids ocs = 1.352  (diagnostic); r = 0.33
cells with valid soil: 96.4% | with MOD17: 96.1%


Wrote data/v2/ludhiana_cells_v2.csv.gz (71,197 cells). 02_carbon_pipeline.ipynb now runs in v2 mode.
